# Nyaya — open-weight teacher: gate first, then RAFT data for v7

Task C4 of the release plan needed a hosted teacher API. This notebook replaces it with an
open model served locally on the Kaggle GPUs (`Qwen/Qwen2.5-14B-Instruct-AWQ`, Apache-2.0,
~10 GB) through vLLM's OpenAI-compatible endpoint, so `scripts/20_generate_raft.py` runs
unchanged against `http://127.0.0.1:8000/v1`.

**The gate comes first.** The teacher answers Eval-v1 under the exact serving prompt
(`scripts/26 --endpoint`, k=8, 768 tokens, the default retriever with `nyaya-embed-v1`, the
same stack as `base-768-embed-v1`). Distillation only makes sense if the teacher is materially
better than the 3B reader it would teach: the paired bootstrap against `base-768-embed-v1` must
exclude zero **and** the fact-recall gain must be at least
5 points. Otherwise the notebook stops and reports that C4 is not worth a training run.

If the gate passes, the teacher regenerates answers for the Nyaya-Train-v3 questions under
the RAFT prompt (gold + distractors, 10% deliberate misses), gated per answer by the context
citation check, 200 val + 1,600 train tasks, written to
`data/generated/nyaya_instruct_v7_raw.jsonl` for `scripts/21` and a v7 LoRA run.

**Settings:** GPU **T4 x2**, Internet On, Input: `jitendrajha98/nyaya-model-src`.
Budget ~4 h (server ~15 min, gate ~40 min, generation ~2 h). No secrets needed.


In [ ]:
# --- setup -------------------------------------------------------------
import glob, json, os, shutil, subprocess, sys, time

# Both T4s stay visible: vLLM tries tensor-parallel 2 first. The only other GPU user in this
# notebook is the e5-base dense stage inside scripts/26 and scripts/20 (one small model).
os.environ["WANDB_DISABLED"] = "true"
os.environ["VLLM_LOGGING_LEVEL"] = "WARNING"

cands = glob.glob("/kaggle/input/nyaya-model-src/**/pyproject.toml", recursive=True)
if not cands:
    import kagglehub
    root = kagglehub.dataset_download("jitendrajha98/nyaya-model-src")
    cands = glob.glob(os.path.join(root, "**", "pyproject.toml"), recursive=True)
assert cands, "dataset jitendrajha98/nyaya-model-src not available to this kernel"
SRC = os.path.dirname(cands[0])
WORK = "/kaggle/working/nyaya-model"
shutil.copytree(SRC, WORK, dirs_exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, "src")

PROGRESS = "/kaggle/working/progress.txt"


def note(msg: str) -> None:
    print(msg, flush=True)
    with open(PROGRESS, "a", encoding="utf-8") as fh:
        fh.write(time.strftime("%H:%M:%S ") + msg + chr(10))


def run(cmd, env=None):
    note("$ " + " ".join(cmd))
    proc = subprocess.run(cmd, text=True, env={**os.environ, **(env or {})})
    if proc.returncode != 0:
        note(f"FAILED exit {proc.returncode}")
        raise RuntimeError(f"step failed (exit {proc.returncode}): {' '.join(cmd)}")
    note("ok")


note(f"setup: source={SRC}")
run([sys.executable, "-m", "pip", "-q", "install", "-r", "requirements-train.txt"])
# vLLM pins its own torch; installing it after the project requirements lets it win.
run([sys.executable, "-m", "pip", "-q", "install", "vllm"])
import vllm  # noqa: E402
note(f"vllm {vllm.__version__}")


In [ ]:
# --- Eval-v1 + GPU preflight -------------------------------------------
import torch

run([sys.executable, "scripts/25_build_eval_v1.py"])
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2.")
for i in range(torch.cuda.device_count()):
    major, minor = torch.cuda.get_device_capability(i)
    note(f"preflight: GPU{i} {torch.cuda.get_device_name(i)} sm_{major}{minor}")
note(f"preflight: torch {torch.__version__}")
assert os.path.exists("outputs/eval-v1/base-768-embed-v1/predictions.jsonl"), "base-768-embed-v1 predictions missing from the snapshot"


In [ ]:
# --- start the teacher server ------------------------------------------------
import requests

TEACHER = os.environ.get("TEACHER", "Qwen/Qwen2.5-14B-Instruct-AWQ")
PORT = 8000
BASE = f"http://127.0.0.1:{PORT}/v1"
SERVER_LOGS = []


def start_server(name, cmd, wait_s=1800, env=None):
    log_path = f"/kaggle/working/server-{name}.log"
    SERVER_LOGS.append(log_path)
    log = open(log_path, "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, **(env or {})})
    t0 = time.time()
    while time.time() - t0 < wait_s:
        if proc.poll() is not None:
            break
        try:
            if requests.get(f"{BASE}/models", timeout=5).status_code == 200:
                note(f"teacher up via {name} after {time.time() - t0:.0f}s")
                return proc
        except requests.RequestException:
            pass
        time.sleep(10)
    if proc.poll() is None:
        proc.kill()
    tail = open(log_path, encoding="utf-8", errors="replace").read().splitlines()[-25:]
    note(f"{name} did not come up; log tail:" + chr(10) + chr(10).join(tail))
    return None


COMMON = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", TEACHER,
          "--served-model-name", TEACHER, "--dtype", "half", "--quantization", "awq",
          "--port", str(PORT), "--enforce-eager"]
# vLLM 0.28 on the T4s: no bf16, no FlashAttention (sm_75). Let the backend auto-select first,
# then force the two Turing-capable V1 backends before giving up.
TP1 = ["--tensor-parallel-size", "1", "--max-model-len", "6144",
       "--gpu-memory-utilization", "0.92", "--max-num-seqs", "8"]
CONFIGS = [
    ("vllm-tp2", COMMON + ["--tensor-parallel-size", "2", "--max-model-len", "8192",
                           "--gpu-memory-utilization", "0.90", "--max-num-seqs", "16"], None),
    ("vllm-tp1", COMMON + TP1, None),
    ("vllm-tp1-triton", COMMON + TP1, {"VLLM_ATTENTION_BACKEND": "TRITON_ATTN"}),
    ("vllm-tp1-flex", COMMON + TP1, {"VLLM_ATTENTION_BACKEND": "FLEX_ATTENTION"}),
]
server = None
for name, cmd, env in CONFIGS:
    server = start_server(name, cmd, env=env)
    if server:
        break
if server is None:
    raise RuntimeError("no teacher server came up; read /kaggle/working/server-*.log")

r = requests.post(f"{BASE}/chat/completions", timeout=300, json={
    "model": TEACHER, "temperature": 0, "max_tokens": 40,
    "messages": [{"role": "user", "content": "Reply with the single word: ready"}]})
note("teacher says: " + r.json()["choices"][0]["message"]["content"].strip()[:80])


In [ ]:
# --- gate: the teacher on Eval-v1 vs the committed base-768-embed-v1 ---------
# Same retriever for both: --dense uses nyaya.dense.DEFAULT_ATTACH_MODEL (nyaya-embed-v1),
# the stack base-768-embed-v1 was scored under.
LABEL = "teacher-" + TEACHER.split("/")[-1].lower()
t0 = time.time()
run([sys.executable, "scripts/26_eval_v1_run.py", "--endpoint", BASE, "--model", TEACHER,
     "--adapter", "none", "--split", "all", "--dense", "--k", "8", "--max-new-tokens", "768",
     "--batch-size", "16", "--concurrency", "16", "--label", LABEL])
note(f"teacher eval done in {(time.time() - t0) / 60:.0f} min")
run([sys.executable, "scripts/27_compare_runs.py", "--a", "base-768-embed-v1", "--b", LABEL])

cmp = json.load(open(f"reports/eval_v1_comparison_{LABEL}.json", encoding="utf-8"))
fr = next(r for r in cmp["results"] if r["metric"] == "fact_recall")
delta, lo, hi = fr["delta"], fr["ci95"][0], fr["ci95"][1]
GATE_PASS = lo > 0 and delta >= 0.05
note(f"GATE {'PASS' if GATE_PASS else 'FAIL'}: teacher fact_recall {fr['b']['mean']:.1%} vs base-768-embed-v1 "
     f"{fr['a']['mean']:.1%}, delta {delta:+.1%}, 95% CI [{lo:+.1%}, {hi:+.1%}] "
     f"(rule: CI excludes zero and delta >= +5 points)")


In [ ]:
# --- RAFT generation (only if the gate passed) ---------------------------------
import random

from huggingface_hub import hf_hub_download

if not GATE_PASS:
    note("skipping RAFT generation: the teacher is not materially better than the 3B reader, "
         "so distillation has no headroom. C4 stays closed.")
else:
    # Nyaya-Train-v3 carries the bare question and gold sections of every training record.
    # Rebuild v1-shaped split files from it; deliberate-miss records are dropped (their gold
    # list is empty by construction) and the order is shuffled so --limit takes a mix of acts.
    os.makedirs("data/splits", exist_ok=True)
    for split in ("train", "val"):
        src = hf_hub_download("NyayaLabs98/nyaya-train-v3", f"{split}.jsonl", repo_type="dataset")
        rows = []
        for line in open(src, encoding="utf-8"):
            rec = json.loads(line)
            meta = rec["metadata"]
            if meta.get("rag", {}).get("is_miss"):
                continue
            rows.append({
                "id": rec["id"],
                "messages": [{"role": "user", "content": meta["rag"]["question"]}],
                "metadata": {k: meta.get(k) for k in ("language", "task_type", "source_act",
                                                      "source_sections", "legal_domain")},
            })
        random.Random(7).shuffle(rows)
        with open(f"data/splits/{split}.jsonl", "w", encoding="utf-8") as fh:
            for r in rows:
                fh.write(json.dumps(r, ensure_ascii=False) + chr(10))
        note(f"splits/{split}.jsonl: {len(rows)} questions from Nyaya-Train-v3")

    env = {"TEACHER_BASE_URL": BASE, "TEACHER_MODEL": TEACHER}
    t0 = time.time()
    run([sys.executable, "scripts/20_generate_raft.py", "--splits", "val", "--limit", "200",
         "--dense", "--concurrency", "16", "--version", "nyaya_instruct_v7"], env=env)
    run([sys.executable, "scripts/20_generate_raft.py", "--splits", "train", "--limit", "1600",
         "--dense", "--concurrency", "16", "--version", "nyaya_instruct_v7"], env=env)
    n_ok = sum(1 for _ in open("data/generated/nyaya_instruct_v7_raw.jsonl", encoding="utf-8"))
    n_rej = sum(1 for _ in open("data/generated/nyaya_instruct_v7_rejected.jsonl", encoding="utf-8")) \
        if os.path.exists("data/generated/nyaya_instruct_v7_rejected.jsonl") else 0
    note(f"RAFT v7: {n_ok} gate-passing answers, {n_rej} rejected, {(time.time() - t0) / 60:.0f} min")


In [ ]:
# --- results: download these --------------------------------------------
import pathlib

server.terminate()
out = pathlib.Path("/kaggle/working/teacher-run")
out.mkdir(exist_ok=True)
for f in ("reports/eval_v1_results.json", f"reports/eval_v1_comparison_{LABEL}.json",
          f"outputs/eval-v1/{LABEL}/predictions.jsonl",
          "data/generated/nyaya_instruct_v7_raw.jsonl", "data/generated/nyaya_instruct_v7_rejected.jsonl"):
    if os.path.exists(f):
        shutil.copy(f, out / (f"{LABEL}_predictions.jsonl" if f.endswith("predictions.jsonl") else os.path.basename(f)))
for log_path in SERVER_LOGS:
    if os.path.exists(log_path):
        shutil.copy(log_path, out)
shutil.make_archive("/kaggle/working/teacher-run", "zip", out)
note("collected: " + ", ".join(sorted(p.name for p in out.iterdir())))
